# Importing Libraries

In [3]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

data = pd.read_csv("Tweets.csv")

# We only need the text and the sentiment column
data = data[['text', 'airline_sentiment']]

print(data.head())
print(data['airline_sentiment'].value_counts())

                                                text airline_sentiment
0                @VirginAmerica What @dhepburn said.           neutral
1  @VirginAmerica plus you've added commercials t...          positive
2  @VirginAmerica I didn't today... Must mean I n...           neutral
3  @VirginAmerica it's really aggressive to blast...          negative
4  @VirginAmerica and it's a really big bad thing...          negative
airline_sentiment
negative    9178
neutral     3099
positive    2363
Name: count, dtype: int64


### The cleaning Function (Regex)

In [6]:
def clean_tweet(text):
    # Remove mentions (@user)
    text = re.sub(r'@[A-Za-z0-9]+', '', text)
    # Remove URLs
    text = re.sub(r'https?://[A-Za-z0-9./]+', '', text)
    # Remove non-letters (numbers and punctuation)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    # Convert to lowercase and strip whitespace
    return text.lower().strip()

# Apply the cleaning 
data['cleaned_text'] = data['text'].apply(clean_tweet)

print("Original:", data['text'][0])
print("Cleaned: ", data['cleaned_text'][0])

Original: @VirginAmerica What @dhepburn said.
Cleaned:  what  said


### Vectorization and Splitting

In [9]:
# Converting the text into numbers using TF-IDF
# Split data 
X_train, X_test, y_train, y_test = train_test_split(data['cleaned_text'], data['airline_sentiment'], test_size=0.2, random_state=42)

# Vectorize
# max_features=5000 means we only keep the top 5,000 most frequent words to keep the model fast
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

### Train and evaluate 

In [11]:
# Logistic Regression is excellent for text classification
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

# Predict
y_pred = model.predict(X_test_vec)

# Evaluate
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.7944

Classification Report:

              precision    recall  f1-score   support

    negative       0.81      0.94      0.87      1889
     neutral       0.68      0.46      0.54       580
    positive       0.82      0.61      0.70       459

    accuracy                           0.79      2928
   macro avg       0.77      0.67      0.70      2928
weighted avg       0.79      0.79      0.78      2928



### Testing with own words

In [12]:
def predict_sentiment(text):
    cleaned = clean_tweet(text)
    vec = vectorizer.transform([cleaned])
    prediction = model.predict(vec)
    return prediction[0]

# Test cases
print("Tweet: 'The flight was amazing and the staff was helpful!'")
print("Prediction:", predict_sentiment("The flight was amazing and the staff was helpful!"))

print("\nTweet: 'I lost my luggage and the service was terrible.'")
print("Prediction:", predict_sentiment("I lost my luggage and the service was terrible."))

Tweet: 'The flight was amazing and the staff was helpful!'
Prediction: positive

Tweet: 'I lost my luggage and the service was terrible.'
Prediction: negative


## Using a pretrained Bert model

In [13]:
! pip install transformers torch

In [14]:
from transformers import pipeline 

In [18]:
# Initialize the sentiment-analysis pipeline
# This downloads a default model (distilbert-base-uncased-finetuned-sst-2-english)
sentiment_pipeline = pipeline("sentiment-analysis")

# Testing
data = [
    "The flight was amazing and the staff was helpful!",
    "I lost my luggage and the service was terrible.",
    "The food was okay, but the seat was uncomfortable." # BERT handles nuance better
]

results = sentiment_pipeline(data)
#printing results
for text, result in zip(data, results):
    print(f"Text: {text}")
    print(f"Label: {result['label']}, Score: {result['score']:.4f}\n")


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

C:\Users\HUMAIDU\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HUMAIDU\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. 

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


Text: The flight was amazing and the staff was helpful!
Label: POSITIVE, Score: 0.9999

Text: I lost my luggage and the service was terrible.
Label: NEGATIVE, Score: 0.9997

Text: The food was okay, but the seat was uncomfortable.
Label: NEGATIVE, Score: 0.9920



In [20]:
!pip install huggingface_hub[hf_xet]